--- 260723 ---

In [ ]:
import os
import sys
import json
import time
import argparse
import random
import glob
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from transformers import AutoTokenizer, AutoConfig
from safetensors.torch import load_file
import pyfaidx
import dotenv
from pathlib import Path
import re

# 导入自定义模块（需确保 src 目录在 Python 路径中）
from src.dataset import MultiTrackDataset, load_fasta_sequence
from src.viewer import DatasetViewer, ResultsViewer
from src.model import GenOmics, load_finetuned_model

# 设置中文字体（根据系统环境调整）
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
class MultiTrackPredictor:
    def __init__(self, fasta_path: str, base_model_path: str, sft_ckpt_path: str,
                 tokenizer_path: str, use_flash_attn: bool, index_stat_path: str,
                 **kwargs):
        self.fasta = pyfaidx.Fasta(fasta_path)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.model = load_finetuned_model(
            model_class = GenOmics,
            model_path = base_model_path,
            ckpt_path = sft_ckpt_path,
            use_flash_attn = use_flash_attn,
            device = "cuda:0",
            model_init_kwargs={"index_stat:" json.load(open(index_stat_path, "r")),
                               **kwargs}
        )
        print("✔️ Model loaded successfully")
        print(self.model)
        self.model.eval()

    def predict2(self, chrom: str, start: int, end: int, seq: str, biosample_names: list = None) -> dict:
        sequence = pyfaidx.Fasta(seq)
        predict_sequence = str(sequence[0][:])
        inputs = self.tokenizer(
            predict_sequence,
            return_tensors="pt",
            padding=False,
            truncation=True,
            max_length=32768,
            add_special_tokens=False
        ).to("cuda")
        with torch.no_grad():
            start_time = time.time()
            result = self.model.predict(input['input_ids'],
                                        biosample_names = biosample_names)
            time_taken = time.time() - start_time
            torch.cuda.empty_cache()
        print(f"Inference time: {time_taken:.2f} s.")
        return {
            'sequence': predict_sequence,
            'position': (chrom, start, end),
            'values': result
        }


In [ ]:
class Args:
    pass

args = Args()
args.mut_fasta = '/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/mutant_predict/riceNavi_output/mutant_fastas/example_214.alt.fa'
args.pattern = '*.alt.fa'
args.output_dir = './mutant_output_260525'
args.save_plots = True
args.plot_tracks = True
args.smoothing_sigma = 10
args.biosample_names = "NIP_Panicle1"

# 固定配置（请根据实际情况修改）
base_model_dir = "/mnt/rice/default/Workspace/xz/hf/rice_1B_stage2_8k_hf"
sft_ckpt_path = "/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/output/202604020731/checkpoint-23540/model.safetensors"
fasta_path = "/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/mutant_predict/osa1_r7.asm.ch.fa"
ANNOTATION_PATH = "/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/mutant_predict/mutant_output_260525/modified_osa1_r7.all_models.gff3"
test_data_dir = "/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/mutant_predict/riceNavi_output"
riceNavi_csv = os.path.join(test_data_dir, "riceNavi.txt")
index_stat_path = "index_stat.json"